# Importe

In [20]:
import os
import shutil
import kagglehub

import pandas as pd

from dfa_rst_pipeline import parse_pair_to_features, discretize_dataframe, quick_reduct, induce_rules

# Daten laden

In [14]:
# Set the path to the file you'd like to load
file_path = "Daten/daten.csv"

# Download latest version
path = kagglehub.dataset_download("sithumigalappaththi/dfa-minimization-data-set")

print("Path to dataset files:", path)

# Finde die heruntergeladene Datei (z. B. eine .csv-Datei)
for file in os.listdir(path):
    if file.endswith(".csv"):
        source_file = os.path.join(path, file)
        break
else:
    raise FileNotFoundError("Keine CSV-Datei im Kaggle-Download-Verzeichnis gefunden.")

# Erstelle Zielordner, falls noch nicht vorhanden
os.makedirs(os.path.dirname(file_path), exist_ok=True)

# Kopiere und benenne um
shutil.copy(source_file, file_path)

print(f"Datei wurde kopiert nach: {file_path}")

Path to dataset files: /home/samel/.cache/kagglehub/datasets/sithumigalappaththi/dfa-minimization-data-set/versions/1
Datei wurde kopiert nach: Daten/daten.csv


# Ausführung

In [21]:
df_train = pd.read_csv("Daten/daten.csv")   # Spalten: input, output
# Feature-Engineering
rows = [parse_pair_to_features(inp, out) for inp, out in zip(df_train["input"], df_train["output"])]
feat = pd.DataFrame(rows)

# Entscheidung: z.B. Reduktionsgrad klassifizieren (nötig für RST)
num_cols = [
    "n_states","alphabet_size","n_transitions","n_self_loops",
    "final_ratio","avg_out_degree","n_reachable","n_unreachable",
    "n_sink_states","reduction_ratio","minimized_n_states"
]
disc = discretize_dataframe(feat, numeric_cols=num_cols, bins=4, strategy="quantile")
cond_attrs = [c for c in disc.columns if c.endswith("_disc") and c != "reduction_ratio_disc"]
decision_attr = "reduction_ratio_disc"

# QuickReduct + einfache Regeln
reduct = quick_reduct(disc, cond_attrs, decision_attr)
rules = induce_rules(disc, reduct, decision_attr)


print("Disc:", disc)

print("Reduct:", reduct)
for r in rules[:10]:
    print(r)


Disc:         n_states  alphabet_size  n_transitions  n_self_loops  final_ratio  \
0              1              2              2             2          0.0   
1              1              2              2             2          1.0   
2              2              2              4             1          1.0   
3              2              2              4             3          0.0   
4              2              2              4             1          0.5   
...          ...            ...            ...           ...          ...   
103735         5              2             10             2          0.4   
103736         5              2             10             2          0.4   
103737         5              2             10             2          0.4   
103738         5              2             10             2          0.2   
103739         5              2             10             2          0.2   

        avg_out_degree  is_complete  n_reachable  n_unreachable  \
0 

In [22]:
df_disc = discretize_dataframe(feat, numeric_cols=num_cols, bins=5, strategy="quantile")
cond_attrs = [c for c in disc.columns if c.endswith("_disc") and c != "reduction_ratio_disc"]
decision_attr = "reduction_ratio_disc"

# QuickReduct + einfache Regeln
reduct = quick_reduct(disc, cond_attrs, decision_attr)
rules = induce_rules(disc, reduct, decision_attr)

print("Reduct:", reduct)
for r in rules[:10]:
    print(r)

print("Rules:", rules)
for r in rules[:10]:
    print(r)

Reduct: []
Rules: []


In [29]:
import pandas as pd
import numpy as np
import re
from collections import defaultdict, Counter, deque
import math
import matplotlib.pyplot as plt
import seaborn as sns


In [23]:
def parse_dfa(text: str):
    parts = [p.strip() for p in text.split(";") if p.strip()]
    transitions = {}
    states = set()
    alphabet = set()
    start = None
    finals = set()

    state_line_pattern = re.compile(r"^([A-Za-z0-9_]+)\s*:\s*(.+)$")
    trans_token_pattern = re.compile(r"\s*([^\s,:]+)\s*-->\s*([^\s,;]+)\s*")

    for p in parts:
        if p.startswith("in:"):
            start = p.split("in:", 1)[1].strip()
        elif p.startswith("fi:"):
            finals = set(s.strip() for s in p.split("fi:", 1)[1].split(",") if s.strip())
        else:
            m = state_line_pattern.match(p)
            if m:
                state = m.group(1).strip()
                states.add(state)
                for tok in m.group(2).split(","):
                    tok = tok.strip()
                    m2 = trans_token_pattern.match(tok)
                    if m2:
                        sym = m2.group(1).strip()
                        tgt = m2.group(2).strip()
                        transitions[(state, sym)] = tgt
                        alphabet.add(sym)
                        states.add(tgt)

    # reachable states
    reachable = set()
    if start:
        queue = [start]
        reachable = {start}
        while queue:
            s = queue.pop()
            for (u, sym), v in transitions.items():
                if u == s and v not in reachable:
                    reachable.add(v)
                    queue.append(v)

    return {
        "states": states,
        "alphabet": alphabet,
        "start": start,
        "finals": finals,
        "transitions": transitions,
        "reachable_states": reachable
    }


In [26]:
def dfa_features(dfa):
    states = dfa["states"]
    alphabet = dfa["alphabet"]
    transitions = dfa["transitions"]
    finals = dfa["finals"]
    reachable = dfa["reachable_states"]
    start = dfa["start"]

    n_states = len(states)
    n_transitions = len(transitions)
    alphabet_size = len(alphabet)
    n_self_loops = sum(1 for (s, a), t in transitions.items() if s == t)
    final_ratio = len(finals)/n_states if n_states else 0

    out_deg = Counter(s for (s,a),t in transitions.items())
    avg_out_degree = np.mean(list(out_deg.values())) if out_deg else 0

    is_complete = all((s,a) in transitions for s in states for a in alphabet)

    n_reachable = len(reachable)
    n_unreachable = n_states - n_reachable

    sink_states = 0
    for s in states:
        if all((s,a) in transitions and transitions[(s,a)]==s for a in alphabet):
            sink_states += 1

    return {
        "n_states": n_states,
        "alphabet_size": alphabet_size,
        "n_transitions": n_transitions,
        "n_self_loops": n_self_loops,
        "final_ratio": final_ratio,
        "avg_out_degree": avg_out_degree,
        "is_complete": int(is_complete),
        "n_reachable": n_reachable,
        "n_unreachable": n_unreachable,
        "n_sink_states": sink_states,
    }

def parse_pair_to_features(inp, out):
    dfa_in = parse_dfa(inp)
    dfa_out = parse_dfa(out)
    f = dfa_features(dfa_in)
    f["minimized_n_states"] = len(dfa_out["states"])
    f["reduction_ratio"] = f["minimized_n_states"]/f["n_states"] if f["n_states"] else np.nan
    return f

def discretize(df, cols, bins=4):
    df = df.copy()
    for c in cols:
        if df[c].nunique() <= 1:
            df[c+"_disc"] = 0
        else:
            df[c+"_disc"] = pd.qcut(df[c].rank(method="first"), q=bins, labels=False, duplicates="drop")
    return df

def indiscernibility(df, attrs):
    blocks = defaultdict(list)
    for i,row in df[attrs].iterrows():
        blocks[tuple(row)] .append(i)
    return list(blocks.values())

def dependency(df, attrs, decision):
    if not attrs:
        return 0
    U = len(df)
    pos = 0
    for block in indiscernibility(df, attrs):
        if len(df.loc[block, decision].unique()) == 1:
            pos += len(block)
    return pos / U

def quick_reduct(df, attrs, decision):
    R = []
    gamma_star = dependency(df, attrs, decision)
    gamma_R = 0

    while gamma_R < gamma_star - 1e-12:
        best_attr = None
        best_gamma = gamma_R
        for a in attrs:
            if a in R: continue
            g = dependency(df, R+[a], decision)
            if g > best_gamma + 1e-12:
                best_gamma = g
                best_attr = a
        if best_attr is None:
            break
        R.append(best_attr)
        gamma_R = best_gamma

    return R

def induce_rules(df, reduct, decision):
    rules = []
    for block in indiscernibility(df, reduct):
        decs = df.loc[block, decision].unique()
        if len(decs)==1:
            rules.append({
                "premise": {a: df.loc[block[0],a] for a in reduct},
                "decision": decs[0],
                "support": len(block),
            })
    return rules


In [38]:
df_train = pd.read_csv("Daten/daten.csv")   # Spalten: input, output

# Feature-Engineering
features = [
    parse_pair_to_features(inp, out) 
    for inp, out in zip(df_train["input"], df_train["output"])
]

# HIER: wirklich DataFrame daraus bauen!
df = pd.DataFrame(features)

num_cols = [
    "n_states","alphabet_size","n_transitions","n_self_loops",
    "final_ratio","avg_out_degree","n_reachable","n_unreachable",
    "n_sink_states","reduction_ratio","minimized_n_states"
]

df_disc = discretize(df, num_cols, bins=4)

decision_attr = "reduction_ratio_disc"
cond_attrs = [c for c in df_disc.columns if c.endswith("_disc") and c != decision_attr]

reduct = quick_reduct(df_disc, cond_attrs, decision_attr)
rules = induce_rules(df_disc, reduct, decision_attr)

print("Reduct:")
print(reduct)

print("\nRules:")
for r in rules[:15]:
    print(r)


Reduct:
[]

Rules:


In [39]:
print("Verteilung der Entscheidung (reduction_ratio_disc):")
print(df_disc[decision_attr].value_counts())

print("\nUnique Counts der Konditionsattribute:")
for c in cond_attrs:
    print(c, "→", df_disc[c].nunique())


Verteilung der Entscheidung (reduction_ratio_disc):
reduction_ratio_disc
3    25935
1    25935
0    25935
2    25935
Name: count, dtype: int64

Unique Counts der Konditionsattribute:
n_states_disc → 4
alphabet_size_disc → 1
n_transitions_disc → 4
n_self_loops_disc → 4
final_ratio_disc → 4
avg_out_degree_disc → 1
n_reachable_disc → 4
n_unreachable_disc → 4
n_sink_states_disc → 4
minimized_n_states_disc → 4


In [41]:
# starke Reduktion = minimierter Automat hat höchstens die Hälfte der Zustände
df["strong_reduction"] = (df["reduction_ratio"] <= 0.5).astype(int)

df_disc = discretize(df, num_cols, bins=4)
decision_attr = "strong_reduction"
cond_attrs = [c for c in df_disc.columns if c.endswith("_disc")]

reduct = quick_reduct(df_disc, cond_attrs, decision_attr)
rules = induce_rules(df_disc, reduct, decision_attr)

print("Reduct:", reduct)
print("Rules:", rules[:10])


Reduct: ['reduction_ratio_disc', 'n_states_disc', 'minimized_n_states_disc', 'n_unreachable_disc', 'final_ratio_disc', 'n_sink_states_disc', 'n_self_loops_disc']
Rules: [{'premise': {'reduction_ratio_disc': 3, 'n_states_disc': 0, 'minimized_n_states_disc': 0, 'n_unreachable_disc': 0, 'final_ratio_disc': 0, 'n_sink_states_disc': 2, 'n_self_loops_disc': 1}, 'decision': 0, 'support': 11}, {'premise': {'reduction_ratio_disc': 3, 'n_states_disc': 0, 'minimized_n_states_disc': 0, 'n_unreachable_disc': 0, 'final_ratio_disc': 3, 'n_sink_states_disc': 2, 'n_self_loops_disc': 1}, 'decision': 0, 'support': 11}, {'premise': {'reduction_ratio_disc': 1, 'n_states_disc': 0, 'minimized_n_states_disc': 0, 'n_unreachable_disc': 0, 'final_ratio_disc': 3, 'n_sink_states_disc': 0, 'n_self_loops_disc': 0}, 'decision': 1, 'support': 110}, {'premise': {'reduction_ratio_disc': 1, 'n_states_disc': 0, 'minimized_n_states_disc': 0, 'n_unreachable_disc': 3, 'final_ratio_disc': 0, 'n_sink_states_disc': 2, 'n_self_l